# RSS LD Sketch Pipeline

## Overview

This pipeline generates a stochastic genotype sample **U = WᵀG** from whole-genome sequencing VCF files and stores it as a PLINK2 pgen file for use as an LD reference panel with SuSiE-RSS fine-mapping.

**Key idea:** Rather than storing the full genotype matrix G (n × p, ~530 MB/block), we compute U = WᵀG (B × p, ~320 MB/block) using a random projection matrix W ~ N(0, 1/√n). The approximate LD matrix R̂ = UᵀU / B ≈ R = GᵀG / n by the Johnson-Lindenstrauss lemma. G is never stored.

**Matrix dimensions:**
- G : (n × p) — n individuals × p variants (~8,000 per block)
- W : (n × B) — projection matrix, W ~ N(0, 1/√n), generated once per cohort
- U : (B × p) — stochastic genotype sample = WᵀG, stored in pgen
- R̂ : (p × p) — approximate LD matrix, computed on-the-fly by SuSiE-RSS from U

**Pipeline steps:**
1. `generate_W` — generate W once from cohort sample size n (saved as `.npy`, shared across all chromosomes)
2. `process_block_1` — read LD block BED file, emit one coordinate file per block
3. `process_block_2` — for each block in parallel: load VCF, compute U = WᵀG, scale to [0,2], write compressed dosage + allele frequencies
4. `merge_chrom` — for each chromosome: convert per-block dosage files to sorted pgen, merge into one per-chromosome pgen, clean up all intermediates

## Input Files

| File | Description |
|------|-------------|
| VCF (.bgz) | Per-chromosome VCF files with tabix index |
| `sample_list.txt` | Optional. One sample ID per line. Leave empty to use all samples. |
| `EUR_LD_blocks.bed` | LD block BED file (0-based half-open, e.g. Berisa & Pickrell EUR blocks) |

## Parameters

**Global parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `plink2_bin` | `plink2` | Path to plink2 binary |
| `walltime` | `24:00:00` | Per-task walltime |
| `mem` | `32G` | Per-task memory |
| `numThreads` | `8` | Per-task CPU cores |

**`generate_W` parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `n_samples` | required | Total number of individuals in the cohort |
| `output_dir` | required | Output directory for `W_B{B}.npy` |
| `B` | `10000` | Sketch dimension (number of pseudo-individuals) |
| `seed` | `123` | Random seed |

**`process_block` parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `vcf_base` | required | Directory containing VCF (.bgz) files |
| `vcf_prefix` | required | Shared filename prefix of all VCF files |
| `ld_block_file` | required | Path to LD block BED file |
| `chrom` | `0` | Chromosome 1–22, or 0 for all chromosomes |
| `cohort_id` | `ADSP.R5.EUR` | Cohort prefix for output filenames |
| `output_dir` | required | Base output directory |
| `W_matrix` | required | Path to W `.npy` output from `generate_W` |
| `B` | `10000` | Must match W |
| `sample_list` | `""` | Optional path to sample ID file |
| `maf_min` | `0.0005` | Minimum minor allele frequency |
| `mac_min` | `5` | Minimum minor allele count |
| `msng_min` | `0.05` | Maximum missingness rate per variant |

**`merge_chrom` parameters:**

| Parameter | Default | Description |
|-----------|---------|-------------|
| `chrom` | `0` | Chromosome 1–22, or 0 for all chromosomes |
| `output_dir` | required | Base output directory (same as `process_block`) |
| `cohort_id` | `ADSP.R5.EUR` | Must match `process_block` cohort_id |

## Output Files

| File | Description |
|------|-------------|
| `W_B{B}.npy` | Projection matrix W, shape (n × B). Output of `generate_W`. |
| `{cohort_id}.chr{N}.pgen` | Final output — per-chromosome stochastic genotype sample U (zstd, plink2 native) |
| `{cohort_id}.chr{N}.pvar.zst` | Per-chromosome variant information (zstd, plink2 native) |
| `{cohort_id}.chr{N}.psam.zst` | Pseudo-individual IDs S1…S{B} (zstd, plink2 native) |
| `{cohort_id}.chr{N}.afreq` | Per-chromosome allele frequencies (OBS_CT = 2 × n) |
| *(cleaned up after merge)* | Per-block `.dosage.gz`, `.map`, `.afreq`, `.meta` intermediate files |


### Step 0 — Generate W (run once per cohort)


In [ ]:
sos run pipeline/rss_ld_sketch.ipynb generate_W \
    --n-samples 16571 \
    --output-dir output/rss_ld_sketch \
    --B 10000 \
    --seed 123


### Step 1+2 — Process all blocks → compressed dosage files


In [ ]:
# Process one chromosome
sos run pipeline/rss_ld_sketch.ipynb process_block \
    --vcf-base /path/to/vcfs/ \
    --vcf-prefix your.prefix. \
    --ld-block-file /path/to/EUR_LD_blocks.bed \
    --chrom 22 \
    --output-dir output/rss_ld_sketch \
    --W-matrix output/rss_ld_sketch/W_B10000.npy \
    --mem 4G \
    -J 16


In [ ]:
# Process all 22 chromosomes
sos run pipeline/rss_ld_sketch.ipynb process_block \
    --vcf-base /restricted/projectnb/xqtl/R4_QC_NHWonly_rm_monomorphic \
    --vcf-prefix gcad.qc.r4.wgs.36361.GATK.2022.08.15.biallelic.genotypes. \
    --ld-block-file /restricted/projectnb/casa/oaolayin/ROSMAP_NIA_geno/EUR_LD_blocks.bed \
    --chrom 0 \
    --output-dir output/rss_ld_sketch \
    --W-matrix output/rss_ld_sketch/W_B10000.npy \
    --mem 4G \
    -J 126


### Step 3 — Merge per-block dosage files into per-chromosome pgen


In [ ]:
sos run pipeline/rss_ld_sketch.ipynb merge_chrom \
    --output-dir output/rss_ld_sketch \
    --chrom 0 \
    -s force


## Command Interface

In [ ]:
sos run pipeline/rss_ld_sketch.ipynb -h

```
usage: sos run pipeline/rss_ld_sketch.ipynb
               [workflow_name | -t targets] [options] [workflow_options]
  workflow_name:        Single or combined workflows defined in this script
  options:              Single-hyphen sos parameters (see "sos run -h" for details)
  workflow_options:     Double-hyphen workflow-specific parameters

Workflows:
  generate_W
  process_block
  merge_chrom

Global Workflow Options:
  --walltime 24:00:00
  --mem 32G
  --numThreads 8 (as int)
  --plink2-bin plink2

Sections
  generate_W:
    Workflow Options:
      --n-samples VAL (as int, required)
                        Generate projection matrix W ~ N(0, 1/sqrt(n)), shape (n x B).
                        Run ONCE per cohort before any chromosome processing.
                        W depends only on n (sample size) and B — not on variant data.
                        All 22 chromosomes reuse the same W.
      --output-dir VAL (as str, required)
      --B 10000 (as int)
      --seed 123 (as int)

  process_block:
    Workflow Options:
      --vcf-base VAL (as str, required)
                        Directory containing VCF (.bgz) files, one or more per chromosome.
      --vcf-prefix VAL (as str, required)
                        Shared filename prefix of all VCF files (everything before "chr{N}:" or "chr{N}.").
      --ld-block-file VAL (as str, required)
                        Path to LD block BED file (e.g. EUR_LD_blocks.bed).
      --chrom 0 (as int)
                        Chromosome 1–22, or 0 for all chromosomes.
      --cohort-id ADSP.R5.EUR (as str)
                        Cohort prefix for output filenames.
      --output-dir VAL (as str, required)
      --W-matrix VAL (as str, required)
      --B 10000 (as int)
      --maf-min 0.0005 (as float)
      --mac-min 5 (as int)
      --msng-min 0.05 (as float)
      --sample-list "" (optional path to sample ID file)
    Output per block (intermediate, cleaned up by merge_chrom):
      {cohort_id}.{chr}_{start}_{end}.dosage.gz
      {cohort_id}.{chr}_{start}_{end}.map
      {cohort_id}.{chr}_{start}_{end}.afreq
      {cohort_id}.{chr}_{start}_{end}.meta

  merge_chrom:
    Workflow Options:
      --chrom 0 (as int)
                        Chromosome 1–22, or 0 for all chromosomes.
      --cohort-id ADSP.R5.EUR (as str)
                        Must match cohort-id used in process_block.
      --output-dir VAL (as str, required)
    Output per chromosome (final, block intermediates cleaned up):
      {cohort_id}.chr{N}.pgen
      {cohort_id}.chr{N}.pvar
      {cohort_id}.chr{N}.psam
      {cohort_id}.chr{N}.afreq
```


# Pipeline Implementation


In [ ]:
[global]
parameter: cwd        = path("output")
parameter: job_size   = 1
parameter: walltime   = "24:00:00"
parameter: mem        = "32G"
parameter: numThreads = 8
parameter: container  = ""
parameter: plink2_bin = "plink2"

import re
from sos.utils import expand_size

entrypoint = (
    'micromamba run -a "" -n' + ' ' +
    re.sub(r'(_apptainer:latest|_docker:latest|\.sif)$', '', container.split('/')[-1])
) if container else ""

cwd = path(f'{cwd:a}')


## `generate_W`

Generates the random projection matrix W ~ N(0, 1/√n), shape (n × B). Run once per cohort. W is shared across all chromosomes.


In [ ]:
[generate_W]
# Generate projection matrix W ~ N(0, 1/sqrt(n)), shape (n x B).
# Run ONCE before processing any chromosome.
#
# W depends only on n (total sample size) and B — not on any variant data.
# n_samples is passed directly as a parameter; no VCF reading is needed.
# All 22 chromosomes reuse the same W so that per-chromosome stochastic
# genotype samples can be arithmetically merged for meta-analysis.
parameter: n_samples = int
parameter: output_dir    = str
parameter: B         = 10000
parameter: seed      = 123

import os
input:  []
output: f'{output_dir}/W_B{B}.npy'
task: trunk_workers = 1, trunk_size = 1, walltime = '00:05:00', mem = '4G', cores = 1
python: expand = "${ }", stdout = f'{_output:n}.stdout', stderr = f'{_output:n}.stderr'

    import numpy as np
    import os

    n      = ${n_samples}
    B      = ${B}
    seed   = ${seed}
    W_out  = "${_output}"

    # ── Generate W ~ N(0, 1/sqrt(n)) ─────────────────────────────
    # Convention: W = np.random.normal(0, 1/np.sqrt(n), size=(n, B))
    # W is shared across all chromosomes — do not regenerate per chromosome.
    print(f"Generating W ~ N(0, 1/sqrt({n})),  shape ({n}, {B}),  seed={seed}")
    np.random.seed(seed)
    W = np.random.normal(0, 1.0 / np.sqrt(n), size=(n, B)).astype(np.float32)

    os.makedirs(os.path.dirname(os.path.abspath(W_out)), exist_ok=True)
    np.save(W_out, W)
    print(f"Saved: {W_out}")
    print(f"Shape: {W.shape},  size: {os.path.getsize(W_out)/1e9:.2f} GB")

## `process_block`

Two-step workflow. `process_block_1` reads the BED file and emits one coordinate file per LD block. `process_block_2` reads the VCF for each block, computes U = WᵀG, scales to [0,2], and writes compressed dosage + allele frequency files. G is never stored.


In [ ]:
[process_block_1]
# Read BED file and write one .block coord file per LD block.
parameter: ld_block_file = str
parameter: chrom         = 0

import os

def _read_blocks(bed, chrom_filter):
    blocks = []
    with open(bed) as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            c = parts[0]
            if not (c.startswith("chr") and c[3:].isdigit()):
                continue
            cnum = int(c[3:])
            if not (1 <= cnum <= 22):
                continue
            if chrom_filter != 0 and cnum != chrom_filter:
                continue
            blocks.append((c, int(parts[1]), int(parts[2])))
    if not blocks:
        raise ValueError(f"No blocks found for chrom={chrom_filter} in {bed}")
    return blocks

_blocks = _read_blocks(ld_block_file, chrom)
print(f"  {len(_blocks)} LD blocks queued")

import tempfile
block_dir = tempfile.mkdtemp(prefix="sos_blocks_")

input:  []
output: [f"{block_dir}/{c}_{s}_{e}.block" for c, s, e in _blocks]
python: expand = "${ }"
    import os, json
    blocks = ${_blocks!r}
    block_dir = "${block_dir}"
    for c, s, e in blocks:
        path = os.path.join(block_dir, f"{c}_{s}_{e}.block")
        with open(path, "w") as fh:
            fh.write(f"{c}\n{s}\n{e}\n")


## `process_block_2`

Per-block parallel task. Reads VCF, computes U = WᵀG in chunks, scales to [0,2], writes `.dosage.gz`, `.map`, `.afreq`, `.meta`. Output is intermediate — cleaned up by `merge_chrom`.


In [ ]:
[process_block_2]
parameter: vcf_base      = str
parameter: vcf_prefix    = str
parameter: cohort_id     = "ADSP.R5.EUR"
parameter: output_dir    = str
parameter: W_matrix      = str
parameter: B             = 10000
parameter: maf_min       = 0.0005
parameter: mac_min       = 5
parameter: msng_min      = 0.05
parameter: sample_list   = ""
input:  output_from("process_block_1"), group_by = 1
task: trunk_workers = 1, trunk_size = 1, walltime = walltime, mem = mem, cores = numThreads
python: expand = "${ }"

    import numpy as np
    import os
    import gzip
    import sys
    from math import nan
    from cyvcf2 import VCF
    from os import listdir

    # Read block coordinates from input file
    with open("${_input}") as fh:
        lines = fh.read().splitlines()
    chrm_str    = lines[0]
    block_start = int(lines[1])
    block_end   = int(lines[2])

    vcf_base    = "${vcf_base}"
    vcf_prefix  = "${vcf_prefix}"
    W_path      = "${W_matrix}"
    B           = ${B}
    maf_min     = ${maf_min}
    mac_min     = ${mac_min}
    msng_min    = ${msng_min}
    sample_list = "${sample_list}"
    cohort_id   = "${cohort_id}"
    base_dir    = "${output_dir}"

    block_tag   = f"{chrm_str}_{block_start}_{block_end}"
    output_dir  = os.path.join(base_dir, chrm_str, block_tag)
    os.makedirs(output_dir, exist_ok=True)

    log_path = os.path.join(output_dir, f"{block_tag}.log")
    log_fh   = open(log_path, "w")
    sys.stdout = log_fh
    sys.stderr = log_fh

    # ── Load sample subset (optional) ─────────────────────────────
    sample_subset = None
    if sample_list:
        if not os.path.exists(sample_list):
            raise FileNotFoundError(f"sample_list not found: {sample_list}")
        with open(sample_list) as fh:
            sample_subset = set(line.strip() for line in fh if line.strip())
        print(f"  Sample subset: {len(sample_subset):,} samples")

    # ── Helpers ───────────────────────────────────────────────────
    def get_vcf_files(chrm_str):
        files = sorted([
            os.path.join(vcf_base, x)
            for x in listdir(vcf_base)
            if x.endswith(".bgz") and (
                x.startswith(vcf_prefix + chrm_str + ":") or
                x.startswith(vcf_prefix + chrm_str + ".")
            )
        ])
        if not files:
            raise FileNotFoundError(f"No VCF files for {chrm_str} in {vcf_base}")
        return files

    def fill_missing_col_means(G):
        col_means = np.nanmean(G, axis=0)
        return np.where(np.isnan(G), col_means, G)

    # ── Step 1: Scan variants in block ────────────────────────────
    print(f"[1/3] Scanning {chrm_str} [{block_start:,}, {block_end:,}) ...")
    vcf_files = get_vcf_files(chrm_str)
    region    = f"{chrm_str}:{block_start+1}-{block_end}"
    var_info  = []
    n_samples = None

    for vf in vcf_files:
        vcf = VCF(vf)
        if sample_subset is not None:
            vcf_samples = vcf.samples
            keep_idx = [i for i, s in enumerate(vcf_samples) if s in sample_subset]
            if not keep_idx:
                raise ValueError(f"No sample_list samples in {os.path.basename(vf)}")
            vcf.set_samples([vcf_samples[i] for i in keep_idx])
        if n_samples is None:
            n_samples = len(vcf.samples)
        for var in vcf(region):
            if not (block_start <= var.POS - 1 < block_end):
                continue
            if len(var.ALT) != 1:
                continue
            dosage = [
                sum(x[0:2])
                for x in [[nan if v == -1 else v for v in gt]
                           for gt in var.genotypes]
            ]
            if np.nanvar(dosage) == 0:
                continue
            counts    = [np.nansum([2 - x for x in dosage]), np.nansum(dosage)]
            nan_count = int(np.sum(np.isnan(dosage)))
            n_non_na  = len(dosage) - nan_count
            mac       = min(counts)
            maf       = mac / n_non_na
            af        = float(np.nansum(dosage)) / (2 * n_non_na)
            msng_rate = nan_count / (n_non_na + nan_count)
            if maf < maf_min or mac < mac_min or msng_rate > msng_min:
                continue
            var_info.append({
                "chr": var.CHROM, "pos": var.POS,
                "ref": var.REF,   "alt": var.ALT[0],
                "af":  round(float(af), 6),
                "id":  f"{var.CHROM}:{var.POS}:{var.REF}:{var.ALT[0]}",
            })
        vcf.close()
    print(f"  {len(var_info):,} variants passing filters  (n={n_samples:,})")

    if not var_info:
        raise ValueError(f"No passing variants in {chrm_str} [{block_start:,}, {block_end:,})")

    # ── Step 2: Load W ────────────────────────────────────────────
    print(f"[2/3] Loading W ...")
    W = np.load(W_path)
    if W.shape != (n_samples, B):
        raise ValueError(f"W shape mismatch: {W.shape} vs ({n_samples},{B})")
    W = W.astype(np.float32)
    print(f"  W: {W.shape}")

    # ── Step 3: Compute U, write compressed dosage + map + afreq ──
    print(f"[3/3] Computing U and writing output files ...")
    var_lookup = {v["id"]: v for v in var_info}
    var_id_set = set(var_lookup)

    dosage_path = os.path.join(output_dir, f"{cohort_id}.{block_tag}.dosage.gz")
    map_path    = os.path.join(output_dir, f"{cohort_id}.{block_tag}.map")
    afreq_path  = os.path.join(output_dir, f"{cohort_id}.{block_tag}.afreq")
    meta_path   = os.path.join(output_dir, f"{cohort_id}.{block_tag}.meta")

    # Write .map
    with open(map_path, "w") as fh:
        for v in var_info:
            fh.write(f"{v['chr']}\t{v['id']}\t0\t{v['pos']}\n")

    # Write .meta
    with open(meta_path, "w") as fh:
        fh.write(f"source_n_samples={n_samples}\nB={B}\n")
        fh.write(f"chrom={chrm_str}\nblock_start={block_start}\nblock_end={block_end}\n")

    # Write .afreq
    obs_ct = 2 * n_samples
    with open(afreq_path, "w") as fh:
        fh.write("#CHROM\tID\tREF\tALT\tALT_FREQS\tOBS_CT\n")
        for v in var_info:
            fh.write(f"{v['chr']}\t{v['id']}\t{v['ref']}\t{v['alt']}\t"
                     f"{v['af']:.6f}\t{obs_ct}\n")

    # Compute U and write compressed dosage (format=1: ID ALT REF val_S1 ... val_SB)
    total = 0
    with gzip.open(dosage_path, "wt", compresslevel=4) as gz:
        for vf in vcf_files:
            arr, var_keys = [], []
            vcf_obj = VCF(vf)
            for var in vcf_obj(region):
                if not (block_start <= var.POS - 1 < block_end):
                    continue
                if len(var.ALT) != 1:
                    continue
                vid = f"{var.CHROM}:{var.POS}:{var.REF}:{var.ALT[0]}"
                if vid not in var_id_set:
                    continue
                dosage = [
                    sum(x[0:2])
                    for x in [[nan if v == -1 else v for v in gt]
                              for gt in var.genotypes]
                ]
                arr.append(dosage)
                var_keys.append(vid)
            vcf_obj.close()
            if not arr:
                continue
            G_chunk = np.array(arr, dtype=np.float32).T
            G_chunk = fill_missing_col_means(G_chunk)
            U_chunk = W.T @ G_chunk
            del G_chunk
            col_min = U_chunk.min(axis=0)
            col_max = U_chunk.max(axis=0)
            denom   = col_max - col_min
            denom[denom == 0] = 1.0
            U_chunk = 2.0 * (U_chunk - col_min) / denom
            U_chunk = np.round(U_chunk, 4)
            for j, vid in enumerate(var_keys):
                v = var_lookup[vid]
                vals = " ".join(f"{x:.4f}" for x in U_chunk[:, j])
                gz.write(f"{vid} {v['alt']} {v['ref']} {vals}\n")
            total += len(var_keys)
            del U_chunk
    print(f"  Written: {total:,} variants → {os.path.basename(dosage_path)}")
    print(f"  Written: {os.path.basename(map_path)}, {os.path.basename(afreq_path)}")
    print(f"\nDone: {chrm_str} [{block_start:,}, {block_end:,})")


## `merge_chrom`

For each chromosome: converts per-block `.dosage.gz` files to sorted pgen (two-pass: import then `--sort-vars`), merges all blocks into one per-chromosome pgen using `plink2 --pmerge-list`, concatenates allele frequencies, and removes all per-block intermediate directories.

Output per chromosome:
- `{cohort_id}.chr{N}.pgen` — binary genotype data (uncompressed — plink2 does not support .pgen compression)
- `{cohort_id}.chr{N}.pvar.zst` — variant info, zstd-compressed via `--make-pgen vzs`
- `{cohort_id}.chr{N}.psam` — sample info (tiny, ~145KB)
- `{cohort_id}.chr{N}.afreq` — allele frequencies


In [ ]:
[merge_chrom]
# Merge all per-block dosage files for a chromosome into one per-chrom pgen.
# Steps per chromosome:
#   1. For each block: plink2 --import-dosage → unsorted pgen, then --sort-vars → sorted pgen
#   2. plink2 --pmerge-list → one per-chrom pgen (.pvar.zst via vzs, .pgen plain)
#   3. Clean up all intermediates including -merge.* files
#   Note: final pgen/pvar/psam are zstd-compressed via --make-pgen vzs (plink2 native)
#
# Run after process_block completes (or run together: sos run ... process_block merge_chrom)
parameter: chrom      = 0
parameter: output_dir = str
parameter: cohort_id  = "ADSP.R5.EUR"

import os, glob

def _chroms_to_process(output_dir, chrom_filter):
    if chrom_filter != 0:
        return [f"chr{chrom_filter}"]
    return sorted(set(
        os.path.basename(d)
        for d in glob.glob(os.path.join(output_dir, "chr*"))
        if os.path.isdir(d)
    ))

_chroms = _chroms_to_process(output_dir, chrom)

input: []
output: [f"{output_dir}/{c}/{cohort_id}.{c}.pgen" for c in _chroms]

for _chrom in _chroms:
    bash(f"""
    set -euo pipefail
    shopt -s nullglob

    chrom_dir="{output_dir}/{_chrom}"
    final_prefix="${{chrom_dir}}/{cohort_id}.{_chrom}"
    merge_list="${{chrom_dir}}/{cohort_id}.{_chrom}_pmerge_list.txt"

    # ── Step 1: Convert each block dosage.gz → sorted per-block pgen (two-pass) ──
    > "${{merge_list}}"
    files=("${{chrom_dir}}"/*/*.dosage.gz)
    if [ ${{#files[@]}} -eq 0 ]; then
        exit 1
    fi
    for dosage_gz in "${{files[@]}}"; do
        block_dir=$(dirname "${{dosage_gz}}")
        block_tag=$(basename "${{block_dir}}")
        prefix="${{block_dir}}/{cohort_id}.${{block_tag}}_tmp"
        map_file="${{block_dir}}/{cohort_id}.${{block_tag}}.map"
        psam_file="${{block_dir}}/{cohort_id}.${{block_tag}}.psam"
        meta_file="${{block_dir}}/{cohort_id}.${{block_tag}}.meta"
        B=$(grep "^B=" "${{meta_file}}" | cut -d= -f2)
        printf '#FID\tIID\n' > "${{psam_file}}"
        for i in $(seq 1 ${{B}}); do
            printf 'S%d\tS%d\n' ${{i}} ${{i}} >> "${{psam_file}}"
        done
        # Pass 1: import dosage → unsorted pgen
        "{plink2_bin}" \
            --import-dosage "${{dosage_gz}}" format=1 noheader \
            --psam "${{psam_file}}" \
            --map  "${{map_file}}" \
            --make-pgen \
            --out  "${{prefix}}_unsorted" \
            --silent
        # Pass 2: sort variants
        "{plink2_bin}" \
            --pfile "${{prefix}}_unsorted" \
            --make-pgen \
            --sort-vars \
            --out  "${{prefix}}" \
            --silent
        rm -f "${{prefix}}_unsorted.pgen" "${{prefix}}_unsorted.pvar" "${{prefix}}_unsorted.psam"
        echo "${{prefix}}" >> "${{merge_list}}"
    done

    # ── Step 2: Merge all per-block pgens → one per-chrom pgen ──
    # vzs compresses .pvar only — .pgen cannot be compressed by plink2 (by design)
    "{plink2_bin}" \
        --pmerge-list "${{merge_list}}" pfile \
        --make-pgen vzs \
        --sort-vars \
        --out  "${{final_prefix}}"

    # ── Step 3: Concatenate .afreq ──
    first=1
    for f in "${{chrom_dir}}"/*/*.afreq; do
        if [ "${{first}}" -eq 1 ]; then
            cat "${{f}}" > "${{final_prefix}}.afreq"
            first=0
        else
            tail -n +2 "${{f}}" >> "${{final_prefix}}.afreq"
        fi
    done

    # ── Step 4: Cleanup — remove merge intermediates and per-block dirs ──
    rm -f "${{merge_list}}"
    rm -f "${{final_prefix}}-merge.pgen" "${{final_prefix}}-merge.pvar" "${{final_prefix}}-merge.psam"
    for block_dir in "${{chrom_dir}}"/*/; do
        rm -rf "${{block_dir}}"
    done

    """)


### Step 4 — Load LD sketch data into R

Load the final pgen output into R using `pecotmr::load_LD_matrix()` via a metadata TSV.

**Metadata TSV format** for PLINK-based LD: one row per chromosome with `start=0`, `end=0`, and `path` as the PLINK file prefix (resolved relative to the TSV directory).

```
#chrom	start	end	path
22	0	0	ADSP.R5.EUR.chr22
```

In [ ]:
library(pecotmr)

meta <- "/path/to/ld_meta_chr22.tsv"
region <- "chr22:30000000-32000000"  # 2Mb region

# Load genotype matrix X (for susie_rss z+X interface)
system.time({
  res_x <- load_LD_matrix(meta, region, return_genotype = TRUE)
})
cat("X dimensions (samples x variants):", dim(res_x$LD_matrix), "\n")
head(res_x$ref_panel, 3)
res_x$LD_matrix[1:3, 1:3]

# Load LD correlation matrix R (for susie_rss z+R interface)
system.time({
  res_r <- load_LD_matrix(meta, region, return_genotype = FALSE)
})
cat("R dimensions:", dim(res_r$LD_matrix), "\n")
res_r$LD_matrix[1:3, 1:3]

**Tested output** (ADSP R5 EUR chr22, 10000 pseudo-samples, 2Mb region, 9188 variants):

```
# Genotype matrix X
   user  system elapsed
  0.680   0.170   0.900

X dimensions (samples x variants): 10000 9188

  chrom      pos A2 A1         variant_id allele_freq
1    22 30000353  G  T chr22:30000353:G:T    0.734539
2    22 30000443  C  T chr22:30000443:C:T    0.001058
3    22 30000447  C  T chr22:30000447:C:T    0.005081

   chr22:30000353:G:T chr22:30000443:C:T chr22:30000447:C:T
S1          0.6563110          0.7753906           1.065613
S2          0.7034302          0.3729858           0.805481
S3          0.8706055          1.1093750           1.046509

# LD correlation matrix R
   user  system elapsed
  6.390   0.410   6.880

R dimensions: 9188 9188
```

Use `return_genotype = TRUE` for the `susie_rss(z, X=X)` interface, or `FALSE` for the `susie_rss(z, R=R)` interface.
For repeated analyses on the same region, compute R once and save with `saveRDS()` for reuse.